# Inference Gate GPU experiment
Start a fresh Kaggle session with T4 GPU and Internet enabled. Run the single cell. Source is embedded; no input upload or kernel restart needed. On any failure the cell prints logs and generates an evidence ZIP.

In [ ]:
ARCHIVE = '...'
import base64, os, shutil, subprocess, sys, tempfile, traceback, zipfile
from pathlib import Path
from IPython.display import FileLink, display

root = Path('/kaggle/working')
work = Path(tempfile.mkdtemp(prefix='inference-gate-run-'))
project = work / 'inference-gate'
results = root / 'inference-gate-results'
if results.exists():
    shutil.move(str(results), str(work / 'previous-results'))
results.mkdir(parents=True, exist_ok=True)
log = results / 'setup-and-run.log'
archive = work / 'inference-gate-source.zip'
venv = work / 'venv'
python = venv / 'bin/python'

def run(command, cwd=None, timeout=1800):
    print('\nRunning:', ' '.join(map(str, command)), flush=True)
    with log.open('a') as out:
        out.write('\nRunning: ' + ' '.join(map(str, command)) + '\n')
        out.flush()
        proc = subprocess.Popen(list(map(str, command)), cwd=cwd, stdout=out, stderr=subprocess.STDOUT, text=True, start_new_session=True)
        try:
            code = proc.wait(timeout=timeout)
        except BaseException:
            import signal
            os.killpg(proc.pid, signal.SIGKILL)
            proc.wait()
            raise
    print('Finished with exit code', code, flush=True)
    if code:
        raise RuntimeError('Command failed with exit code ' + str(code))

success = False
try:
    archive.write_bytes(base64.b64decode(ARCHIVE))
    with zipfile.ZipFile(archive) as z:
        assert all((work / item.filename).resolve().is_relative_to(work.resolve()) for item in z.infolist())
        z.extractall(work)
    assert (project / 'scripts/kaggle_run.py').is_file()
    run(['nvidia-smi'], timeout=30)
    import ctypes
    driver = ctypes.CDLL('libcuda.so.1')
    loaded_paths = [line.split(maxsplit=5)[-1].strip() for line in Path('/proc/self/maps').read_text().splitlines()]
    driver_path = next(Path(p) for p in loaded_paths if p.startswith('/') and Path(p).name.startswith('libcuda.so') and Path(p).is_file())
    links = work / 'nvidia-link'
    links.mkdir()
    (links / 'libcuda.so').symlink_to(driver_path.resolve())
    os.environ['LIBRARY_PATH'] = os.pathsep.join(filter(None, (str(links), os.environ.get('LIBRARY_PATH'))))
    probe = work / 'link-probe.cc'
    probe.write_text('extern "C" int cuInit(unsigned); int main() { return cuInit(0); }')
    run(['c++', probe, '-lcuda', '-o', work / 'link-probe'], timeout=60)
    run([work / 'link-probe'], timeout=30)
    run([sys.executable, '-m', 'venv', '--without-pip', str(venv)], timeout=120)
    run([sys.executable, '-m', 'pip', '--python', str(python), 'install', '--no-cache-dir', 'wrapt', 'pip', '-e', str(project), 'vllm==0.11.2'], timeout=1800)
    run([python, '-c', 'import torch,vllm; print("torch",torch.__version__,"vllm",vllm.__version__,"CUDA",torch.cuda.is_available()); assert torch.cuda.is_available(), "Select T4 GPU and restart the Kaggle session"; print("GPU",torch.cuda.get_device_name(0),"capability",torch.cuda.get_device_capability(0)); assert torch.cuda.get_device_capability(0)[0] >= 7, "T4 or newer required"'], timeout=120)
    run([python, 'scripts/kaggle_run.py'], cwd=project, timeout=5400)
    success = True
except Exception:
    print('\nEXPERIMENT FAILED — details follow:', flush=True)
    traceback.print_exc()
    with log.open('a') as out:
        traceback.print_exc(file=out)
finally:
    engine = results / 'engine.log'
    if log.exists():
        print('\n===== SETUP AND RUN LOG: last 15000 characters =====', flush=True)
        print(log.read_text(errors='replace')[-15000:], flush=True)
    for gateway_log in sorted(results.glob('rate-*/*.log')):
        if gateway_log.stat().st_size:
            print('\n===== ' + str(gateway_log.relative_to(results)) + ' =====', flush=True)
            print(gateway_log.read_text(errors='replace')[-3000:], flush=True)
    if engine.exists():
        print('\n===== SERVER LOG: last 15000 characters =====', flush=True)
        print(engine.read_text(errors='replace')[-15000:], flush=True)
    else:
        print('\nNo server log was created; inspect setup-and-run.log above.', flush=True)
    (results / 'run-status.txt').write_text('completed\n' if success else 'FAILED: partial results, not a completed experiment\n')
    output = shutil.make_archive(str(root / 'inference-gate-gpu-evidence'), 'zip', results)
    print('\nDownload evidence:', output, flush=True)
    os.chdir(root)
    display(FileLink(Path(output).name))
if not success:
    print('Benchmark failed; diagnostic output saved.')
